In [7]:
import json
import pandas as pd

In [8]:
# Load the dependencies JSON file
json_path = '../data/dependencies.json'

with open(json_path, 'r', encoding='utf-8') as f:
    dependencies_data = json.load(f)

print(f"Loaded {len(dependencies_data)} course entries")

Loaded 136 course entries


In [9]:
def extract_requirement_of(link_with):
    """Extract RequirementOf codes as a semicolon-separated string"""
    if not link_with or 'RequirementOf' not in link_with:
        return ''
    
    requirement_of = link_with['RequirementOf']
    if not requirement_of:
        return ''
    
    # Extract codes from the RequirementOf list
    codes = [req['Code'] for req in requirement_of if 'Code' in req]
    return '; '.join(codes)

def transform_to_csv_row(course):
    """Transform a course JSON object into a CSV row dictionary"""
    return {
        'Subject': course.get('Subject', ''),
        'Code': course.get('Code', ''),
        'Type': course.get('Type', ''),
        'Semester': course.get('Semester', ''),
        'RequirementOf': extract_requirement_of(course.get('LinkWith', {})),
        'Goal': course.get('About', {}).get('Goal', ''),
        'About': course.get('About', {}).get('Program', '')
    }

# Transform all courses
csv_data = [transform_to_csv_row(course) for course in dependencies_data]

# Create DataFrame
df = pd.DataFrame(csv_data)

print(f"Created DataFrame with {len(df)} rows and {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Created DataFrame with 136 rows and 7 columns

Columns: ['Subject', 'Code', 'Type', 'Semester', 'RequirementOf', 'Goal', 'About']

First few rows:


,Subject,Code,Type,Semester,RequirementOf,Goal,About
0,Informação Profissional e Tutoria Acadêmica em...,SCC0200,Disciplinas Obrigatórias,1º Semestre Ideal,,Ambientar o estudante com o curso de Bacharela...,"Disciplina de caráter informativo, oferecendo ..."
1,Introdução à Ciência de Computação I,SCC0221,Disciplinas Obrigatórias,1º Semestre Ideal,SCC0201,Apresentar os conceitos básicos para o desenvo...,Conceitos básicos sobre computadores: hardware...
2,Laboratório de Introdução à Ciência de Computa...,SCC0222,Disciplinas Obrigatórias,1º Semestre Ideal,,Implementar em laboratório as técnicas de prog...,Resolução de problemas e desenvolvimento de pr...
3,Geometria Analítica,SMA0300,Disciplinas Obrigatórias,1º Semestre Ideal,SMA0394,Visa familiarizar os alunos com a geometria an...,Coordenadas cartesianas. Vetores. Dependência...
4,Cálculo I,SMA0353,Disciplinas Obrigatórias,1º Semestre Ideal,7600105; SMA0354; SMA0392,Fazer com que os alunos familiarizem-se com os...,O conjunto dos números reais. Funções Reais de...


In [10]:
# Display basic info
print("DataFrame Info:")
print(df.info())

print("\n" + "="*80)
print("\nValue counts by Type:")
print(df['Type'].value_counts())

print("\n" + "="*80)
print("\nValue counts by Semester:")
print(df['Semester'].value_counts().sort_index())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Subject        136 non-null    object
 1   Code           136 non-null    object
 2   Type           136 non-null    object
 3   Semester       136 non-null    object
 4   RequirementOf  136 non-null    object
 5   Goal           136 non-null    object
 6   About          136 non-null    object
dtypes: object(7)
memory usage: 7.6+ KB
None


Value counts by Type:
Type
Disciplinas Optativas Eletivas    83
Disciplinas Obrigatórias          46
Disciplinas Optativas Livres       7
Name: count, dtype: int64


Value counts by Semester:
Semester
10º Semestre Ideal     4
1º Semestre Ideal      9
2º Semestre Ideal     11
3º Semestre Ideal      9
4º Semestre Ideal      9
5º Semestre Ideal     11
6º Semestre Ideal     10
7º Semestre Ideal     33
8º Semestre Ideal     35
9º Semestre Ideal   

In [11]:
# Check for courses with RequirementOf relationships
courses_with_requirements = df[df['RequirementOf'] != '']
print(f"\nCourses that are requirements for other courses: {len(courses_with_requirements)}")
print("\nExamples:")
courses_with_requirements[['Code', 'Subject', 'RequirementOf']].head(10)


Courses that are requirements for other courses: 35

Examples:


,Code,Subject,RequirementOf
1,SCC0221,Introdução à Ciência de Computação I,SCC0201
3,SMA0300,Geometria Analítica,SMA0394
4,SMA0353,Cálculo I,7600105; SMA0354; SMA0392
7,SSC0117,Introdução à Lógica Digital,SSC0118
9,7600105,Física Básica I,7600109
10,SCC0201,Introdução à Ciência de Computação II,SCC0215; SCC0215; SCC0205; SCC0230; SCC0210; S...
11,SCC0202,Algoritmos e Estruturas de Dados I,SCC0216; SSC0103; SSC0960; SCC0282; SCC0284
12,SMA0180,Matemática Discreta I,SCC0283; SCC0283
13,SMA0354,Cálculo II,SMA0355; SME0123; SMA0390
15,SSC0118,Sistemas Digitais,SSC0902


In [12]:
# Save to CSV
output_path = '../data/dependencies.csv'
df.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Data successfully saved to: {output_path}")
print(f"   - Rows: {len(df)}")
print(f"   - Columns: {list(df.columns)}")

✅ Data successfully saved to: ../data/dependencies.csv
   - Rows: 136
   - Columns: ['Subject', 'Code', 'Type', 'Semester', 'RequirementOf', 'Goal', 'About']
